**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**05 | Participant-Balanced Descriptive Statistics**

![Conceptual workflow for stage 05](../../images/05_participant_balanced_statistics.svg)

**Objective:** Summarise recorded gaze features by task with uncertainty based on participants rather than samples.

**Dataset:** Current Notebook 04 per-record features and Notebook 01 quality summary.

**Notebook scope:** Descriptive, exploratory statistics only; no unwarranted group-comparison p-values.

**Execution status:** Source code prepared. No original-data results are claimed until this notebook is executed with the specified release.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 5. Study Research Utilities ------------------------
from core.stage_outputs import ensure_stage_inputs
from gaze_source import read_stage
from gaze_statistics import participant_summary, summarise_participant_metrics

# --------------------- 6. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


**02. Recover Feature and Source Quality Tables**

Use existing record-level features and quality summaries rather than
recomputing raw signal transformations or detector predictions.


In [ ]:
# =============== 02. Recover Feature and Source Quality Tables ===============

ensure_stage_inputs(output_dir, 1, ("01_quality_by_record.csv",))
ensure_stage_inputs(output_dir, 4, ("04_features_by_record.csv",))
quality = read_stage(output_dir/"01_quality_by_record.csv",
                     ("record_id","participant_id","valid_fraction"))
features = read_stage(output_dir/"04_features_by_record.csv",
                      ("record_id","participant_id","task","valid_fraction"))


**03. Aggregate Within Participants**

A participant with repeated recordings does not count as independent
people. Aggregate participant-by-task before reporting descriptive averages.
Feature availability is handled per feature; absent pupils remain missing.


In [ ]:
# ===================== 03. Aggregate Within Participants =====================

METRICS=("valid_fraction","mean_fixation_ms","scanpath_length_dva",
         "mean_saccade_peak_dva_s")
participant = participant_summary(features, METRICS)
participant.to_csv(output_dir/"05_participant_features.csv",index=False)
display(participant)


**04. Calculate Cluster-Level Exploratory Uncertainty**

Use an exploratory 500-resample bootstrap over participant values within
each task, not individual eye samples. When fewer than two participants
provide a feature, uncertainty bounds are undefined and left blank.
Any interval based on very few people must be interpreted cautiously.


In [ ]:
# ============ 04. Calculate Cluster-Level Exploratory Uncertainty ============

stats = summarise_participant_metrics(participant,METRICS,draws=500,seed=17)
stats.to_csv(output_dir/"05_descriptive_statistics.csv",index=False)
display(stats)


**05. Plot Participant-Weighted Descriptive Findings**

Display valid gaze proportion by task with participant-level uncertainty.
No comparative significance or causal interpretation is implied.


In [ ]:
# ============ 05. Plot Participant-Weighted Descriptive Findings =============

valid=stats[stats.feature.eq("valid_fraction")].sort_values("task")
fig,ax=plt.subplots(figsize=(8,4))
x=np.arange(len(valid))
ax.bar(x,valid.participant_mean)
ax.set_xticks(x,valid.task)
ax.set(ylabel="Participant-weighted valid fraction",ylim=(0,1),
       title="Gaze quality by task (exploratory)")
fig.tight_layout()
save_figure(fig,output_dir/"05_statistics.png")
plt.show()


**06. Scientific Interpretation and Download**

Task differences may reflect stimuli, device conditions or record selection.
This study cannot infer cognitive outcomes or generalise from a bounded
convenience subset. Save outputs for the synthesis stage.


In [ ]:
# ================ 06. Scientific Interpretation and Download =================

print("Participant-balanced summaries are descriptive and not validation of a classifier.")
download_outputs_from_colab("eye_tracking_foundations/output","05")
